# Introduction to Tetrad

Tetrad is a Java library of various interfaces for exploring causal explanations of data. With the py-tetrad python interface and JPype, we can access all of Tetrad's functionality, from creating graphs of relationships within data to simulating data from constructed models.

## Prerequisites
- Java JDK 21+ installed on your system
- Python 3.12+
- py-tetrad for basic Tetrad functionality
- JPype for basic and extended Tetrad functionality
- PyTorch for the CausalPerceptronNetwork, used to simulate data from existing graphs

In [3]:
%pip install -U JPype1 git+https://github.com/cmu-phil/py-tetrad torch

  Cloning https://github.com/cmu-phil/py-tetrad to c:\users\devin\appdata\local\temp\pip-req-build-qbu2uhv8
  Resolved https://github.com/cmu-phil/py-tetrad to commit 568b9f5aae98a4cebf20530a9a3dbbbd7e72f13a
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Note: you may need to restart the kernel to use updated packages.


  Running command git clone --filter=blob:none --quiet https://github.com/cmu-phil/py-tetrad 'C:\Users\devin\AppData\Local\Temp\pip-req-build-qbu2uhv8'


## Imports

In [1]:
from importlib import resources as impresources
from pytetrad import resources
import pytetrad.tools.TetradSearch as ts
import edu.cmu.tetrad.data as td
import pandas as pd

## Knowledge

In [3]:
# Reading sample data taken from py-tetrad
resource_file = impresources.files(resources) / 'aw-fb-pruned18.data.mixed.numeric.txt'
knowledge_df = pd.read_csv(resource_file, sep='\t')

knowledge_df.head(5)

,age,gender,height,weight,steps,heart_rate,calories,distance,entropy_heart,entropy_steps,resting_heart,corr_heart_steps,norm_heart,sd_norm_heart,steps_times_distance,device,activity
0,20.0,1.0,168.0,65.4,10.7714,78.5313,0.3445,0.0083,6.2216,6.1163,59.0,1.0000,19.5313,1.0000,0.0897,0.0,0.0
1,20.0,1.0,168.0,65.4,11.4753,78.4534,3.2876,0.0089,6.2216,6.1163,59.0,1.0000,19.4534,1.0000,0.1021,0.0,0.0
2,20.0,1.0,168.0,65.4,12.1792,78.5408,9.4840,0.0095,6.2216,6.1163,59.0,1.0000,19.5408,1.0000,0.1153,0.0,0.0
3,20.0,1.0,168.0,65.4,12.8831,78.6283,10.1546,0.0100,6.2216,6.1163,59.0,1.0000,19.6283,1.0000,0.1293,0.0,0.0
4,20.0,1.0,168.0,65.4,13.5870,78.7157,10.8251,0.0106,6.2216,6.1163,59.0,0.9828,19.7157,0.2416,0.1441,0.0,0.0


We know that factors like age, gender, height, weight, resting heart rate, and whether the user has a device or recorded activity are all determined prior to the activity test recorded in this data. As such, we can include these factors in Tier 0 and the rest in later tiers. Additionally, we can indicate to the model that these factors should not affect those within the same tier.

In [4]:
# Declaring Knowledge tiers
tier0 = ['age', 'gender', 'height', 'weight', 'resting_heart', 'device', 'activity']
tier1 = ['steps', 'heart_rate', 'calories', 'distance']

# Adding tiers to knowledge
knowledge = td.Knowledge()
for col in tier0:
    knowledge.addToTier(0, col)
for col in tier1:
    knowledge.addToTier(1, col)

# Forbid edges within tier0
knowledge.setTierForbiddenWithin(0, True)

As examples, we can specify required and forbidden edges.

In [5]:
# Required edge
knowledge.setRequired('activity', 'steps')

# Forbidden edge
knowledge.setForbidden('age', 'device')

## TetradSearch
To demonstrate use of TetradSearch, first we will run the FGES algorithm on a continuous dataset. The FGES algorithm produces a Completed Partially Directed Acyclic Graph (CPDAG).

In [6]:
# Reading sample data from py-tetrad
resource_file = impresources.files(resources) / 'airfoil-self-noise.continuous.txt'
continuous_df = pd.read_csv(resource_file, sep='\t')
continuous_df = continuous_df.astype({col: "float64" for col in continuous_df.columns})

# Instantiating TetradSearch
search = ts.TetradSearch(continuous_df)

# Adding knowledge through TetradSearch
search.set_forbidden('Frequency', 'Attack')
search.set_required('Frequency', 'Pressure')

# Specify SEM BIC scoring
search.use_sem_bic(penalty_discount=2)

# Run the search -- These are default FGES search parameters
search.run_fges(symmetric_first_step=False, max_degree=-1, parallelized=False, faithfulness_assumed=False)
print(search.get_string())

Dec 13, 2025 12:17:02 AM java.util.prefs.FileSystemPreferences$1 run
INFO: Created user preferences directory.
Dec 13, 2025 12:17:02 AM java.util.prefs.FileSystemPreferences$6 run


Error computing BIC: Graph must not be null.
here
Graph Nodes:
Frequency;Attack;Chord;Velocity;Displacement;Pressure

Graph Edges:
1. Attack --> Frequency
2. Attack --> Pressure
3. Chord --> Attack
4. Chord --> Frequency
5. Chord --> Pressure
6. Displacement --> Attack
7. Displacement --- Chord
8. Displacement --> Pressure
9. Frequency --> Pressure
10. Velocity --> Attack
11. Velocity --> Frequency
12. Velocity --> Pressure

Graph Attributes:
Score: -46106.834126

Graph Node Attributes:
Score: [Frequency: -28323.59197296753;Attack: -7882.65059720316;Chord: 2856.9619390766175;Velocity: -12518.376781557072;Displacement: 8815.0969551796;Pressure: -9054.27366892865]



The error displayed in the above output, "Error computing BIC," is an issue with the py-tetrad library that appears on any FGES-based search. The score attributes indicate the strength of the graph and its constituent nodes, with higher values indicating higher confidence in the determined graph.

To analyze temporal relationships, we can use the time lag parameter to include previous data's effect on current data. Below, we see how a time lag of 1 affects the causal graph by including variables from one record prior. Further increasing the time lag parameter significantly increases the complexity of the model.

In [7]:
# Set time lag parameter
search.set_time_lag(1)

# Run the search again
search.run_fges()
print(search.get_string())

hereError computing BIC: Graph must not be null.

Graph Nodes:
Frequency;Attack;Chord;Velocity;Displacement;Pressure;Frequency:1;Attack:1;Chord:1;Velocity:1;Displacement:1;Pressure:1

Graph Edges:
1. Attack --> Displacement
2. Attack --> Frequency
3. Attack --> Pressure
4. Chord --> Pressure
5. Frequency --> Pressure
6. Velocity --> Frequency
7. Velocity --> Pressure
8. Attack:1 --> Attack
9. Attack:1 --> Displacement
10. Attack:1 --> Frequency:1
11. Attack:1 --> Pressure:1
12. Chord:1 --> Chord
13. Chord:1 --> Pressure
14. Chord:1 --> Attack:1
15. Chord:1 --- Displacement:1
16. Chord:1 --> Frequency:1
17. Chord:1 --> Pressure:1
18. Displacement:1 --> Displacement
19. Displacement:1 --> Attack:1
20. Displacement:1 --> Pressure:1
21. Frequency:1 --> Frequency
22. Frequency:1 --> Pressure
23. Frequency:1 --> Pressure:1
24. Pressure:1 --> Pressure
25. Velocity:1 --> Pressure
26. Velocity:1 --> Velocity
27. Velocity:1 --> Attack:1
28. Velocity:1 --> Frequency:1
29. Velocity:1 --> Pressure:

TetradSearch similarly includes access to algorithms for discrete or mixed datasets. Below is an example of using TetradSearch to run the RFCI algorithm. RFCI is designed for discrete datasets and returns a Partial Ancestral Graph (PAG).

In [8]:
# Reading sample data from py-tetrad
resource_file = impresources.files(resources) / 'bridges.data.version211_rev.txt'
discrete_df = pd.read_csv(resource_file, sep='\t')

# Instantiating TetradSearch
search = ts.TetradSearch(discrete_df)

# Select tests and scores
search.use_bdeu(sample_prior=10, structure_prior=0)
search.use_chi_square(alpha=0.05)

# Run the search
search.run_rfci(depth=-1, stable_fas=True, max_disc_path_length=-1, complete_rule_set_used=True)
print(search.get_string())

Graph Nodes:
RIVER;ERECTED;PURPOSE;LENGTH;LANES;CLEAR_G;T_OR_D;MATERIAL;SPAN;REL_L;TYPE

Graph Edges:
1. CLEAR_G o-> TYPE
2. LANES o-> T_OR_D
3. LENGTH o-> SPAN
4. MATERIAL o-> TYPE
5. REL_L o-> SPAN
6. T_OR_D <-> TYPE




## Running Tetrad Directly with JPype
Some of Tetrad's functionality is not included directly in py-tetrad and must instead be called through JPype. One search algorithm that TetradSearch does not include is the IMaGES algorithm, which is a modified version of the FGES algorithm that can input multiple similarly structured datasets to produce a single CPDAG. The below example uses the IMaGES-BOSS algorithm, which utilizes BOSS instead of FGES and is specifically suited for continuous variables.

In [8]:
import pytetrad.tools.translate as ptt
import edu.cmu.tetrad.util as util
import edu.cmu.tetrad.algcomparison.algorithm.multi as multi
import java.util as jutil

# Reading sample data from py-tetrad
resource_file = impresources.files(resources) / 'airfoil-self-noise.continuous.txt'
continuous_df = pd.read_csv(resource_file, sep='\t')
continuous_df = continuous_df.astype({col: "float64" for col in continuous_df.columns})

# Instantiate the IMaGES-BOSS algorithm 
alg = multi.ImagesBoss()

# Set parameters. Images uses SEM BIC for scoring by default
params = util.Parameters()
params.set(util.Params.PENALTY_DISCOUNT, 2)
params.set(util.Params.RANDOM_SELECTION_SIZE, 1)

# Convert dataframe to Tetrad data object
tetrad_data = ptt.pandas_data_to_tetrad(continuous_df)

# Add data to an ArrayList 
data_list = jutil.ArrayList()
# In this example we will simply reuse the same dataset but different, similarly-structured data also works
data_list.add(tetrad_data)
data_list.add(tetrad_data)
data_list.add(tetrad_data)

# Run and print resultss
cpdag = alg.search(data_list, params)
ptt.print_java(cpdag)

Graph Nodes:
Frequency;Attack;Chord;Velocity;Displacement;Pressure

Graph Edges:
1. Attack --> Displacement
2. Attack --> Pressure
3. Chord --> Attack
4. Chord --> Displacement
5. Chord --> Pressure
6. Displacement --> Pressure
7. Frequency --> Attack
8. Frequency --> Pressure
9. Velocity --> Attack
10. Velocity --- Frequency
11. Velocity --> Pressure




## Simulating Data with Tetrad

### Simulating Continuous Data

In [9]:
import pytetrad.tools.simulate as sim

# Simulate data with its true graph
d_continuous, g_continuous = sim.simulateContinuous(num_meas = 20, num_lat = 0, avg_deg = 4, samp_size = 200, 
                                                    coef_low = 0, coef_high = 1, var_low = 1, var_high = 3, 
                                                    rand_cols=False)

# View true graph
ptt.print_java(g_continuous)

# View data with pandas
df = ptt.tetrad_data_to_pandas(d_continuous)
df.head(5)

Graph Nodes:
X1;X2;X3;X4;X5;X6;X7;X8;X9;X10;X11;X12;X13;X14;X15;X16;X17;X18;X19;X20

Graph Edges:
1. X1 --> X2
2. X1 --> X6
3. X1 --> X10
4. X1 --> X18
5. X2 --> X7
6. X2 --> X15
7. X2 --> X17
8. X2 --> X18
9. X2 --> X19
10. X2 --> X20
11. X3 --> X8
12. X3 --> X10
13. X3 --> X13
14. X3 --> X16
15. X4 --> X17
16. X5 --> X6
17. X6 --> X13
18. X6 --> X14
19. X6 --> X20
20. X7 --> X8
21. X7 --> X9
22. X7 --> X13
23. X7 --> X14
24. X7 --> X19
25. X7 --> X20
26. X8 --> X10
27. X8 --> X14
28. X8 --> X16
29. X8 --> X17
30. X8 --> X18
31. X9 --> X10
32. X9 --> X19
33. X10 --> X16
34. X11 --> X13
35. X11 --> X19
36. X12 --> X18
37. X13 --> X16
38. X15 --> X16
39. X16 --> X20
40. X17 --> X20




,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,X12,X13,X14,X15,X16,X17,X18,X19,X20
0,1.014882,0.647943,2.697061,-0.805802,0.712116,0.388134,0.00036,-2.183603,1.122185,2.26193,-0.342984,1.072056,-0.499727,0.820472,0.179431,-1.397464,0.63739,1.662782,-1.969389,-1.608181
1,2.523997,-0.459881,2.698961,-2.131934,-2.784253,3.854213,-0.000793,-5.899484,-2.973324,2.166657,0.139534,-0.868043,-1.258878,5.061159,-0.719674,-1.887148,-2.829719,-0.140519,-0.692611,-5.298104
2,-0.109398,0.550266,0.699899,0.783145,-0.762392,1.29493,-0.58589,-0.083683,0.201568,-0.495457,0.866878,-2.07483,0.812824,0.026551,-2.205955,0.595463,-0.077902,0.488888,0.205349,-0.145733
3,0.054549,0.661255,0.219238,1.972489,0.824144,-0.694806,-1.206307,1.619452,0.372424,-1.876362,-0.688558,0.537668,-0.265301,-2.121993,0.990722,1.994667,2.439981,0.709003,-0.519532,1.608235
4,-0.82892,1.674332,-2.044415,-1.045375,-1.263684,0.929137,1.23231,1.538445,-0.613188,-3.455253,1.78288,-0.046829,-0.046704,1.062218,-0.778615,-1.683923,0.71273,2.983352,4.306665,0.534854


### Simulating Discrete Data 

In [10]:
# Simulate data with its true graph
d_discrete, g_discrete = sim.simulateDiscrete(num_meas = 20, num_lat = 0, avg_deg = 4, min_cat=3, max_cat=3, 
                                                  samp_size=1000)

# View true graph
ptt.print_java(g_discrete)

# View data with pandas
df = ptt.tetrad_data_to_pandas(d_discrete)
df.head(5)

Graph Nodes:
X1;X2;X3;X4;X5;X6;X7;X8;X9;X10;X11;X12;X13;X14;X15;X16;X17;X18;X19;X20

Graph Edges:
1. X1 --> X10
2. X1 --> X17
3. X1 --> X18
4. X1 --> X19
5. X2 --> X6
6. X2 --> X10
7. X2 --> X13
8. X2 --> X18
9. X2 --> X19
10. X3 --> X12
11. X3 --> X13
12. X3 --> X17
13. X4 --> X11
14. X4 --> X12
15. X5 --> X9
16. X5 --> X11
17. X5 --> X15
18. X5 --> X18
19. X6 --> X14
20. X6 --> X16
21. X7 --> X8
22. X7 --> X10
23. X7 --> X13
24. X7 --> X18
25. X7 --> X20
26. X8 --> X10
27. X8 --> X12
28. X8 --> X14
29. X9 --> X10
30. X9 --> X14
31. X10 --> X13
32. X11 --> X17
33. X12 --> X18
34. X12 --> X20
35. X13 --> X14
36. X13 --> X16
37. X14 --> X15
38. X16 --> X20
39. X17 --> X19
40. X18 --> X19




,X19,X11,X10,X4,X5,X18,X9,X7,X20,X12,X16,X17,X6,X1,X8,X15,X13,X3,X2,X14
0,(0),(0),(1),(0),(0),(2),(1),(0),(1),(0),(0),(2),(1),(1),(1),(1),(1),(0),(1),(2)
1,(0),(1),(2),(2),(0),(1),(1),(2),(1),(1),(1),(2),(0),(2),(2),(2),(2),(0),(1),(1)
2,(2),(0),(1),(0),(0),(2),(0),(1),(0),(0),(1),(0),(0),(2),(1),(0),(2),(0),(1),(0)
3,(1),(2),(1),(0),(1),(1),(1),(1),(1),(0),(1),(2),(1),(2),(2),(1),(0),(2),(0),(2)
4,(0),(0),(1),(0),(0),(0),(1),(1),(2),(0),(2),(0),(0),(2),(2),(2),(0),(0),(1),(0)


### Simulating Mixed Data with Lee & Hastie
The [Lee & Hastie method](https://proceedings.mlr.press/v31/lee13a.html) for simulating mixed datasets treats mixtures of continuous and discrete data as log-linear.

In [11]:
# Simulate data with its true graph
d_mixed, g_mixed = sim.simulateLeeHastie(num_meas = 20, num_lat = 0, avg_deg = 4, min_cat=3, max_cat=3, 
                                         perc_disc=50, samp_size=1000)

# View true graph
ptt.print_java(g_mixed)

# View data with pandas
df = ptt.tetrad_data_to_pandas(d_mixed)
df.head(5)

Graph Nodes:
X1;X2;X3;X4;X5;X6;X7;X8;X9;X10;X11;X12;X13;X14;X15;X16;X17;X18;X19;X20

Graph Edges:
1. X1 --> X8
2. X2 --> X11
3. X2 --> X13
4. X3 --> X7
5. X3 --> X8
6. X3 --> X10
7. X3 --> X11
8. X3 --> X13
9. X3 --> X15
10. X4 --> X9
11. X4 --> X12
12. X4 --> X17
13. X4 --> X18
14. X4 --> X19
15. X5 --> X6
16. X5 --> X9
17. X5 --> X16
18. X6 --> X9
19. X6 --> X18
20. X7 --> X13
21. X7 --> X19
22. X8 --> X11
23. X8 --> X13
24. X8 --> X15
25. X8 --> X19
26. X9 --> X10
27. X9 --> X11
28. X9 --> X12
29. X9 --> X17
30. X9 --> X18
31. X10 --> X18
32. X12 --> X16
33. X12 --> X19
34. X12 --> X20
35. X13 --> X14
36. X13 --> X16
37. X13 --> X18
38. X14 --> X18
39. X14 --> X19
40. X16 --> X18




,X17,X5,X9,X18,X11,X1,X4,X15,X10,X7,X12,X19,X2,X3,X6,X20,X14,X16,X8,X13
0,0.715506,-1.683723,1,3.668629,1,1,0.192125,1.566975,-2.039367,1,1,2,0.861386,1,2,1.79843,0,0.683186,1,-0.362282
1,1.792233,-0.611453,2,2.843084,0,1,-0.30178,3.681569,-3.850738,2,1,2,-1.963663,2,1,1.52257,0,-1.965474,2,-1.273463
2,0.693617,0.881224,2,-0.778894,0,0,-0.254202,-0.960339,-0.971053,1,2,2,-2.886417,1,0,-0.165773,2,-1.813692,1,3.482943
3,2.405043,-1.363623,0,-3.466531,0,0,-0.204007,0.89207,-0.47406,2,2,0,0.093355,0,1,-1.73845,0,-4.423695,2,-2.615626
4,0.183983,-0.021937,0,-0.817973,1,0,-0.710854,-0.18008,-0.162061,1,1,0,2.549953,2,0,0.196491,0,-3.600205,1,-0.964621


### Simulating Data from an Existing Graph
Py-tetrad provides the CausalPerceptronNetwork class based on PyTorch for simulating data based on an input graph. This can be useful for assessing the robustness of a generated graph.

In [12]:
import pytetrad.tools.cpn as cpn
import torch.nn as nn

# Setup a CausalPerceptronNetwork to simulate data from the CPDAG from above
noise_distributions = {}
for node in cpdag.getNodes():
    noise_distributions[node] = cpn.NoiseDistribution(distribution_type="normal", mean=0, std=1)

cpn = cpn.CausalPerceptronNetwork(
    graph=cpdag,
    num_samples=10000,
    noise_distributions=noise_distributions,  # Function to generate noise
    hidden_dimensions=[50, 50, 50, 50, 50],
    input_scale=1,
    activation_module=nn.LeakyReLU(),
    nonlinearity='leaky_relu',
    discrete_prob=0,  # No discrete variables
    seed=40
)

# Simulate the data from the CPN that was created from the PAG
simulated_df = cpn.generate_data()

simulated_df.head(5)

,Frequency,Attack,Chord,Velocity,Displacement,Pressure
0,0.134287,2.102219,1.113105,-0.574891,0.119222,-0.546117
1,0.064648,1.728880,1.890632,-1.656146,0.766208,-0.692765
2,0.078281,0.823285,1.111294,-1.202268,1.664933,1.220293
3,0.096521,0.803707,0.751750,-0.620423,2.059050,-0.404596
4,0.089519,1.287818,-0.844076,-0.862712,0.675245,-0.200326
